# 04 Sequence Preparation for Deep Learning

This notebook prepares the vehicle-level sequence datasets for the deep learning experiments.

The input sequences have already been constructed from the preprocessed operational histories. This notebook applies the final transformations required before model training, including:

1. forward-fill imputation within each sequence;
2. zero-fill imputation for missing values at the beginning of a sequence;
3. categorical encoding of static vehicle features;
4. normalization of numerical sequence features;
5. log transformation of histogram-bin features.

The prepared sequences are used in two modelling stages:

1. self-supervised or semi-supervised representation learning;
2. downstream survival fine-tuning for time-to-failure prediction.

Variable sequence lengths are not forced to a fixed length in this notebook. Instead, sequence alignment for mini-batch training is handled later by the dataset collate function, which pads sequences within each batch and creates the corresponding masks.

## Inputs

- sequence objects generated from the preprocessing stage;
- vehicle-level time-to-event information;
- static vehicle specification features, where applicable.

## Outputs

- imputed and transformed sequence datasets;
- encoded static feature representations;
- normalization parameters estimated from the training data;
- sequence files prepared for SSL pretraining and survival fine-tuning.

## Sequence Transformation Strategy

The sequence data are transformed without removing the temporal structure of each vehicle history. Missing values are handled by forward-fill imputation, followed by zero-fill imputation for values that remain missing at the beginning of a sequence. Numerical features are normalized using statistics estimated from the training set only. Histogram-bin features are log-transformed to reduce scale effects while preserving their accumulated-count interpretation.

The sequences remain variable in length after these transformations. Padding and masking are handled during batch construction by the dataset collate function.

In [ ]:
# import required libraries
import pickle
from pathlib import Path

import numpy as np
import pandas as pd


In [ ]:
# Set project paths and enable imports from the src directory

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
SEQUENCE_DIR = PROJECT_ROOT / "data" / "processed" / "sequences"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

print(f"Notebook location: {Path.cwd()}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Source directory: {SRC_DIR}")

### Load raw truncated sequences

In [ ]:
# load raw truncated sequences (unnormalized)
from config import X_TRAIN_TRUNC_RAW_SEQUENCE_FILE, X_VAL_TRUNC_RAW_SEQUENCE_FILE, X_TEST_TRUNC_RAW_SEQUENCE_FILE
with open(X_TRAIN_TRUNC_RAW_SEQUENCE_FILE, "rb") as f:
    train_seq_truncated = pickle.load(f)

print(f"Number of vehicles: {len(train_seq_truncated)}")

In [ ]:
with open(X_VAL_TRUNC_RAW_SEQUENCE_FILE, "rb") as f:
    val_seq_truncated = pickle.load(f)

print(f"Number of vehicles: {len(val_seq_truncated)}")

In [ ]:
with open(X_TEST_TRUNC_RAW_SEQUENCE_FILE, "rb") as f:
    test_seq_truncated = pickle.load(f)

print(f"Number of vehicles: {len(test_seq_truncated)}")

### Load full raw sequences

In [ ]:
from config import X_TRAIN_FULL_RAW_SEQUENCE_FILE, X_VAL_FULL_RAW_SEQUENCE_FILE, X_TEST_FULL_RAW_SEQUENCE_FILE
# full train sequences
with open(X_TRAIN_FULL_RAW_SEQUENCE_FILE, "rb") as f:
    train_seq = pickle.load(f)

print(f"Number of vehicles: {len(train_seq)}")

In [ ]:
# load the full test raw sequence dictionary
with open(X_TEST_FULL_RAW_SEQUENCE_FILE, "rb") as f:
    test_seq = pickle.load(f)

print(f"Number of vehicles: {len(test_seq)}")

In [ ]:
# load the full validation raw sequence dictionary
with open(X_VAL_FULL_RAW_SEQUENCE_FILE, "rb") as f:
    val_seq = pickle.load(f)

print(f"Number of vehicles: {len(val_seq)}")

### Missing value imputation, feature-aware normalization
Missing values are imputed using forward-fill and zero-fill at the beginning.
Numerical features are normalized with mean and std from the training sequences
Histogram features transformed with logarithmic transformation to preserve skewness and monotonicity (accumulation)

In [ ]:
# for full sequences
from sequence_builder import preprocess_sequence_splits

train_seq_norm, val_seq_norm, test_seq_norm, seq_normalizer = preprocess_sequence_splits(
    train_seq=train_seq, 
    val_seq=val_seq,
    test_seq=test_seq
)

In [ ]:
# for truncated sequences
train_seq_trunc_norm, val_seq_trunc_norm, test_seq_trunc_norm, trunc_seq_normalizer = preprocess_sequence_splits(
    train_seq=train_seq_truncated, 
    val_seq=val_seq_truncated,
    test_seq=test_seq_truncated
)

In [ ]:
# verify if the imputation, normalization and log transformation have worked
print(len(train_seq_norm), len(val_seq_norm), len(test_seq_norm))
print(seq_normalizer.keys())

first_vehicle = next(iter(train_seq_norm))
print(train_seq_norm[first_vehicle].keys())
print(train_seq_norm[first_vehicle]["numerical_sequence"].shape)
print(train_seq_norm[first_vehicle]["histogram_sequence"].shape)

In [ ]:
# verify if the imputation, normalization and log transformation have worked for truncated sequences
print(len(train_seq_trunc_norm), len(val_seq_trunc_norm), len(test_seq_trunc_norm))
print(trunc_seq_normalizer.keys())

first_vehicle_trunc = next(iter(train_seq_trunc_norm))
print(train_seq_trunc_norm[first_vehicle_trunc].keys())
print(train_seq_trunc_norm[first_vehicle_trunc]["numerical_sequence"].shape)
print(train_seq_trunc_norm[first_vehicle_trunc]["histogram_sequence"].shape)

In [ ]:
# print one truncated sequence
train_seq_trunc_norm[first_vehicle_trunc]

In [ ]:
# extra check on whether normalization behaves as intended
all_num = np.concatenate(
    [seq["numerical_sequence"] for seq in train_seq_norm.values()],
    axis=0
)

print("NaNs:", np.isnan(all_num).sum())
print("Mean (approx 0):", all_num.mean(axis=0)[:5])
print("Std (approx 1):", all_num.std(axis=0)[:5])

### One-hot-encode static features

In [ ]:
# one hot encode static features
from sklearn.preprocessing import OneHotEncoder


def fit_static_encoder(sequence_dict):
    static_rows = [
        sample["static_features"]
        for sample in sequence_dict.values()
    ]

    static_df = pd.DataFrame(static_rows)

    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False,
    )

    encoder.fit(static_df)
    return encoder, static_df.columns.tolist()


def add_encoded_static_features(sequence_dict, encoder, static_cols):
    new_dict = {}

    for vehicle_id, sample in sequence_dict.items():
        sample_copy = sample.copy()

        static_df = pd.DataFrame(
            [sample["static_features"]],
            columns=static_cols,
        )

        encoded_static = encoder.transform(static_df)[0].astype(np.float32)

        sample_copy["static_features_encoded"] = encoded_static
        new_dict[vehicle_id] = sample_copy

    return new_dict

In [ ]:
# 1. Full sequences
# fit on train set, use the fitted encoder to transform the validation and test sets
static_encoder, static_cols = fit_static_encoder(train_seq_norm)

train_full_sequences = add_encoded_static_features(
    train_seq_norm,
    static_encoder,
    static_cols,
)

val_full_sequences = add_encoded_static_features(
    val_seq_norm,
    static_encoder,
    static_cols,
)

test_full_sequences = add_encoded_static_features(
    test_seq_norm,
    static_encoder,
    static_cols,
)

In [ ]:
# 2. Truncated sequences
static_encoder_trunc, static_cols_trunc = fit_static_encoder(train_seq_trunc_norm)

train_truncated_sequences = add_encoded_static_features(
    train_seq_trunc_norm,
    static_encoder_trunc,
    static_cols_trunc,
)

val_truncated_sequences = add_encoded_static_features(
    val_seq_trunc_norm,
    static_encoder_trunc,
    static_cols_trunc,
)

test_truncated_sequences = add_encoded_static_features(
    test_seq_trunc_norm,
    static_encoder_trunc,
    static_cols_trunc,
)

In [ ]:
# Save preprocessed full sequences and normalizer
from sequence_builder import save_sequence_artifacts

save_sequence_artifacts(
    train_seq=train_full_sequences,
    val_seq=val_full_sequences,
    test_seq=test_full_sequences,
    normalizer=seq_normalizer,
    encoder=static_encoder,
    output_dir=SEQUENCE_DIR
)

In [ ]:
from sequence_builder import save_truncated_sequence_artifacts
save_truncated_sequence_artifacts(
    train_trunc_seq=train_truncated_sequences,
    val_trunc_seq=val_truncated_sequences,
    test_trunc_seq=test_truncated_sequences,
    normalizer_trunc=trunc_seq_normalizer,
    encoder_trunc=static_encoder_trunc,
    output_dir=SEQUENCE_DIR
)